<img src="logo-facty.jpeg" width="150" align="left"/>

<div style="text-align: center;">
    <h2><b>University of Carabobo</b></h2>
    <h3><b>Experimental Faculty of Sciences and Technology (FACYT)</b></h3>
    <h4><b>Computer Science Department</b></h4>
    <p style="text-align:center"><b>Course:</b> Machine Learning (Elective)<br>
    <b>Professor:</b> Álvaro Espinoza<br>
    <b>Date:</b> August 2026<br>
    <b>Students:</b> Juan Suárez and Josue Carrion</p>
</div>

<br clear="all">

---
# **Data Preparation and Feature Engineering**
## **StarCraft 2 Player Skill Prediction**

### **1. Problem Context and Purpose**
Before building the data preparation pipeline, it is necessary to establish the boundaries of the problem based on the Exploratory Data Analysis (EDA) findings:

*   **Unit of Observation:** A single player's performance metrics and telemetry recorded during a specific StarCraft 2 ranked match.
*   **Target Variable:** `LeagueIndex` (Integer ranging from 1 to 8, where 1 is Bronze and 8 is Professional). This frames our task as a Multiclass (and inherently Ordinal) Classification problem.
*   **Prediction Meaning:** A prediction represents the estimated competitive rank or skill tier of a player, derived strictly from their mechanical speed, attention shifting, and macro-economy management during a single game.
*   **Information Available at Prediction Time:** We assume the prediction occurs *after* a match has concluded and the replay file has been parsed. Therefore, all system-generated in-game telemetry (e.g., `APM`, `ActionLatency`, `NumberOfPACs`) is fully available and accurate. Self-reported demographic data (`Age`, `HoursPerWeek`) is also available but subject to voluntary disclosure (as seen with the missing data in League 8).
*   **Excluded Columns:** 
    *   `GameID`: Must be strictly excluded. It is a surrogate key / unique identifier for the match. It carries no mechanical or cognitive predictive signal and keeping it would only introduce noise or risk overfitting (memorization of IDs by the model).

### **2. Connecting EDA Findings to Preparation Decisions**

Every preprocessing step in this pipeline is strictly motivated by the evidence gathered during the Exploratory Data Analysis.

| EDA Finding | Evidence | Decision | Justification |
| :--- | :--- | :--- | :--- |
| **Missing Not At Random (MNAR)** | 100% of League 8 players lack demographic data (`Age`, `TotalHours`, etc.). | Apply `SimpleImputer(strategy='median')` to demographic variables **without** adding a missing indicator. | Creating a missing indicator would introduce **Target Leakage**, acting as a "cheat code" for League 8 due to dataset anonymization artifacts. Imputing with the median forces the model to predict League 8 strictly based on in-game mechanical telemetry. |
| **Physically Impossible Demographics** | Max `TotalHours` = 1,000,000; Max `HoursPerWeek` = 168. | Apply filtering to drop these specific rows before the split. | These are data entry errors that would skew scaling and model weights without providing valid information. |
| **Extreme Mechanical Telemetry** | High outliers in `APM` (up to ~389) and `NumberOfPACs`. | Keep outliers; apply `RobustScaler` or `StandardScaler` in the pipeline. | These are valid representations of the mechanical ceiling of elite professionals, not errors. Dropping them would destroy the signal for League 8. |
| **Severe Multicollinearity** | Strong inverse correlation (r = -0.72) between `APM` and `ActionLatency`. | Engineer `CognitiveSpeedIndex` feature: `APM / (ActionLatency + 1)`. | Provides a simplified, composite metric to reduce dimensionality for algorithms sensitive to multicollinearity. |
| **Target Class Imbalance** | Leagues 1, 7 and 8 are heavily underrepresented compared to middle leagues. | Use `stratify=y` during the `train_test_split`. | Ensures that players are proportionally distributed in both training and test sets. |

### **3. Imports and Constants definition**

In [ ]:
# Standard Library
import os
import numpy as np
import pandas as pd

# Scikit-Learn: Configuration
from sklearn import set_config

# Scikit-Learn: Model Selection
from sklearn.model_selection import train_test_split

# Scikit-Learn: Pipeline and Preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, FunctionTransformer

# --- NEW: Global Scikit-Learn Configuration ---
# Forces transformers to output Pandas DataFrames instead of Numpy Arrays
set_config(transform_output="pandas")

# --- Configuration Constants ---
RANDOM_SEED = 42
DATA_PATH = '../data/SkillCraft1_Dataset.csv'
TARGET_COL = 'LeagueIndex'
ID_COL = 'GameID'

# StarCraft 2 telemetry conversion constant (1 timestamp = 1/88.5 seconds)
TIMESTAMPS_PER_MINUTE = 88.5 * 60

# List of all features expected by the pipeline
REQUIRED_FEATURES = [
    'Age', 'HoursPerWeek', 'TotalHours', 'APM', 'SelectByHotkeys', 
    'AssignToHotkeys', 'UniqueHotkeys', 'MinimapAttacks', 
    'MinimapRightClicks', 'NumberOfPACs', 'GapBetweenPACs', 
    'ActionLatency', 'ActionsInPAC', 'TotalMapExplored', 
    'WorkersMade', 'UniqueUnitsMade', 'ComplexUnitsMade', 
    'ComplexAbilitiesUsed'
]

print("Libraries imported. Pandas output configured. Constants defined.")

Libraries imported. Pandas output configured. Constants defined.


### **4. Data Loading and Structural Validation**
In this section, we load the raw data and perform basic sanity checks (presence of target variable, duplicate identifiers, and valid categories). We also apply the deterministic filtering defined in our EDA table to remove data entry errors *before* the train/test split.

In [2]:
# Load dataset
df = pd.read_csv(DATA_PATH, na_values=['?'])

print("--- DATA VALIDATION ---")

# 1. Check for Required Feature Columns
missing_cols = [col for col in REQUIRED_FEATURES if col not in df.columns]
if len(missing_cols) == 0:
    print(f"[OK] All {len(REQUIRED_FEATURES)} required feature columns are present.")
else:
    raise ValueError(f"[ERROR] Missing required columns: {missing_cols}")

# 2. Check for Target Variable
if TARGET_COL in df.columns:
    print(f"[OK] Target variable '{TARGET_COL}' found.")
else:
    raise ValueError(f"[ERROR] Target variable '{TARGET_COL}' is missing!")

# 3. Check for Duplicate Identifiers
if ID_COL in df.columns:
    duplicates = df[ID_COL].duplicated().sum()
    if duplicates == 0:
        print(f"[OK] No duplicate identifiers found in '{ID_COL}'.")
    else:
        print(f"[WARNING] Found {duplicates} duplicate identifiers.")
else:
    raise ValueError(f"[ERROR] Identifier column '{ID_COL}' is missing!")

# 4. Target Variable Invalid Categories Validation
# LeagueIndex should only contain integers from 1 to 8
invalid_target_cats = df[~df[TARGET_COL].isin(range(1, 9))]
if len(invalid_target_cats) == 0:
    print(f"[OK] All '{TARGET_COL}' values are within valid categories (1-8).")
else:
    print(f"[WARNING] Found {len(invalid_target_cats)} invalid categories in target variable.")

print("\n--- DETERMINISTIC FILTERING (As defined in EDA) ---")
initial_shape = df.shape[0]

# Ensure demographic columns are numeric
for col in ['Age', 'HoursPerWeek', 'TotalHours']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Filter physically impossible values while explicitly NaNs to protect League 8
df = df[(df['TotalHours'] < 50000) | (df['TotalHours'].isna())]
df = df[(df['HoursPerWeek'] < 168) | (df['HoursPerWeek'].isna())]

filtered_shape = df.shape[0]
rows_dropped = initial_shape - filtered_shape

print(f"[ACTION] Dropped {rows_dropped} rows due to physically impossible demographic values.")
print(f"[CHECK] League 8 representation preserved: {df[df[TARGET_COL] == 8].shape[0]} elite players remaining.")
print(f"[INFO] Dataset shape ready for Train/Test Split: {df.shape}")

--- DATA VALIDATION ---
[OK] All 18 required feature columns are present.
[OK] Target variable 'LeagueIndex' found.
[OK] No duplicate identifiers found in 'GameID'.
[OK] All 'LeagueIndex' values are within valid categories (1-8).

--- DETERMINISTIC FILTERING (As defined in EDA) ---
[ACTION] Dropped 2 rows due to physically impossible demographic values.
[CHECK] League 8 representation preserved: 55 elite players remaining.
[INFO] Dataset shape ready for Train/Test Split: (3393, 20)


### **5. Train / Test Split and Leakage Prevention**
To prevent data leakage, the dataset is split *before* any statistical transformations are adjusted. 

*   **Split Strategy Justification:** Given the severe class imbalance observed during the EDA (particularly the underrepresentation of Leagues 1, 7 and 8), we employ **Stratified Sampling** (`stratify=y`) based on the target variable. This ensures that the rare professional players are proportionally distributed into both the training and testing sets, rather than being accidentally clumped into one.
*   **Data Leakage Acknowledgement:** It is important to acknowledge that our initial EDA was performed on the entire dataset. Therefore, the test set has indirectly influenced some structural decisions (like discovering the MNAR pattern in League 8). However, from this point forward, the test set is strictly isolated. All imputers and scalers will be fitted exclusively on the training data.

In [3]:
print("--- TRAIN / TEST SPLIT ---")

# Separate features (X) and target (y)
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Perform stratified split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.20, 
    random_state=RANDOM_SEED, 
    stratify=y
)

print(f"[OK] Split successful. Random seed used: {RANDOM_SEED}")
print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")

# 1. Verify partitions do not share observations using the unique identifier (GameID)
shared_obs = len(set(X_train[ID_COL]).intersection(set(X_test[ID_COL])))
if shared_obs == 0:
    print(f"\n[OK] Verification passed: Training and testing sets do NOT share any observations.")
else:
    raise ValueError(f"[ERROR] Data Leakage detected: {shared_obs} shared observations found between Train and Test!")

# 2. Verify stratification (check the proportion of League 8)
train_l8_prop = (y_train == 8).mean() * 100
test_l8_prop = (y_test == 8).mean() * 100

print(f"\n[CHECK] Stratification Verification (League 8 representation):")
print(f"Training set League 8 proportion: {train_l8_prop:.2f}%")
print(f"Testing set League 8 proportion:  {test_l8_prop:.2f}%")

--- TRAIN / TEST SPLIT ---
[OK] Split successful. Random seed used: 42
X_train shape: (2714, 19) | y_train shape: (2714,)
X_test shape:  (679, 19)  | y_test shape:  (679,)

[OK] Verification passed: Training and testing sets do NOT share any observations.

[CHECK] Stratification Verification (League 8 representation):
Training set League 8 proportion: 1.62%
Testing set League 8 proportion:  1.62%


### **6. Feature Engineering (Custom Transformer)**
Following the assignment requirements, we engineer features belonging to distinct families (Ratios, Aggregations, Booleans, and Log Transforms). To ensure this is understandable for stakeholders unfamiliar with StarCraft 2, we justify each feature using both EDA findings and specific eSports domain knowledge.

**1. Strategic Variables (Domain Knowledge):**
*   **`CognitiveSpeedIndex` (Ratio):** `APM / (ActionLatency + 1)`. In StarCraft, APM (Actions Per Minute) measures hand speed, while Action Latency measures mental reaction time. High APM with slow reactions usually means mindless "spam" clicking. This index condenses both into a single "meaningful speed" metric, isolating players who are both fast and precise.
*   **`ComplexMechanicsRate` (Aggregation) & `UsesComplexUnits` (Boolean):** Beginners rely on basic armies that attack automatically. Elite players use "spellcasters" (complex units)—fragile units that require intense manual clicking to activate special abilities mid-battle. These variables act as direct proxies for high-level "micro-management".
*   **`MinimapUsage` (Aggregation):** `MinimapAttacks + MinimapRightClicks`. Novices move their camera by dragging the edges of the screen. Professionals issue commands directly on the tiny radar (minimap) to manage multiple battles across the map simultaneously without looking away from their main base. This captures global "macro-awareness".
*   **`WeeksOfPlay` (Ratio):** `TotalHours / HoursPerWeek`. Approximates the player's seniority or tenure in the game, differentiating a veteran who plays casually from a newcomer who plays intensely.

**2. Skewness Reduction & Unit Conversion:**
*   **The Statistical Problem (EDA Finding):** Our EDA identified severe right-skewness across time-related features and action-rate variables (`TotalHours` skew > 50). To compress these massive statistical tails and assist linear models, a log transform (`np.log1p`) is required.
*   **The Technical Solution (Domain Knowledge):** Applying a log transform directly to the raw in-game rates (`MinimapAttacks`, `SelectByHotkeys`, etc.) is mathematically ineffective because they are recorded per *game-engine timestamp* (~1/88.5 seconds), resulting in minuscule values (~1e-4) where `log(1 + x) ≈ x`. Using StarCraft 2 architecture knowledge, we fix this by first multiplying these rates by `88.5 * 60` (converting them to human-readable "actions per minute") *before* applying the log transform.
*   To prevent redundant representations, the raw, highly skewed columns are explicitly dropped from the dataset inside the transformer.

In [4]:
print("--- FEATURE ENGINEERING PREPARATION ---")

# Define columns that require log transformation to reduce extreme right-skewness
HOUR_COLS = ["TotalHours", "HoursPerWeek"]
RATE_COLS = ["MinimapAttacks", "MinimapRightClicks", "SelectByHotkeys", "ComplexAbilitiesUsed"]
LOG_COLS = HOUR_COLS + RATE_COLS

def engineer_features(X):
    """
    Stateless feature engineering function. 
    Applies domain ratios, boolean rules, and skewness reduction.
    """
    X_out = X.copy()
    
    # 1. Ratios, Sums, and Booleans
    X_out['CognitiveSpeedIndex'] = X_out['APM'] / (X_out['ActionLatency'] + 1)
    X_out['MinimapUsage'] = X_out['MinimapAttacks'] + X_out['MinimapRightClicks']
    X_out['ComplexMechanicsRate'] = X_out['ComplexUnitsMade'] + X_out['ComplexAbilitiesUsed']
    
    # Calculate seniority. Replace 0 with NaN to avoid DivisionByZero before log transform
    X_out['WeeksOfPlay'] = X_out['TotalHours'] / X_out['HoursPerWeek'].replace(0, np.nan)
    X_out['UsesComplexUnits'] = (X_out['ComplexUnitsMade'] > 0).astype(int)
    
    # 2. Log Transforms to fix skewness
    for c in HOUR_COLS:
        X_out[f"log_{c}"] = np.log1p(X_out[c])
        
    for c in RATE_COLS:
        # Convert timestamps to per-minute rate, then apply log1p
        X_out[f"log_{c}"] = np.log1p(X_out[c] * TIMESTAMPS_PER_MINUTE)
        
    X_out["log_WeeksOfPlay"] = np.log1p(X_out.pop("WeeksOfPlay"))
    
    # 3. Cleanup: Drop the raw versions of the log-transformed columns
    X_out = X_out.drop(columns=LOG_COLS)
    
    # Clean up any infinites generated by divisions
    return X_out.replace([np.inf, -np.inf], np.nan)

# Wrap the function in a Scikit-Learn FunctionTransformer
feature_engineer_transformer = FunctionTransformer(func=engineer_features, validate=False)

print("[OK] Super Custom Feature Engineering Transformer created successfully.")

--- FEATURE ENGINEERING PREPARATION ---
[OK] Super Custom Feature Engineering Transformer created successfully.


### **7. Feature Grouping and Organization**
As required, we must explicitly define the analytical roles of our variables before building the preprocessing pipelines. Since our `engineer_features` function dynamically creates and drops columns, we will generate a temporary preview of the transformed data to accurately extract and group the final feature names.

*   **Numeric with Missing:** Variables that require imputation (`Age`, and our new `log_` time variables which may contain `NaN`s from the physical outlier filter).
*   **Standard Numeric:** All other continuous or discrete telemetry variables.
*   **Binary:** Our newly engineered `UsesComplexUnits` indicator.
*   **Excluded Columns:** `GameID` will be explicitly dropped as it is a surrogate key holding no predictive value.

In [5]:
print("--- 7. VARIABLE ORGANIZATION ---")

# Generate a temporary preview to extract dynamic column names after engineering
X_fe_preview = feature_engineer_transformer.transform(X_train)

# 1. Excluded Columns
EXCLUDED = [ID_COL]

# 2. Binary Features
BINARY = ["UsesComplexUnits"]

# 3. Numeric Features with Missing Data (MNAR demographics or generated NaNs)
NUMERIC_WITH_MISSING = ["Age", "log_TotalHours", "log_HoursPerWeek", "log_WeeksOfPlay"]

# 4. Standard Numeric Features (Everything else)
NUMERIC = [c for c in X_fe_preview.columns if c not in EXCLUDED + BINARY + NUMERIC_WITH_MISSING]

print(f"Excluded               ({len(EXCLUDED):2d}): {EXCLUDED}")
print(f"Binary                 ({len(BINARY):2d}): {BINARY}")
print(f"Numeric with Missing   ({len(NUMERIC_WITH_MISSING):2d}): {NUMERIC_WITH_MISSING}")
print(f"Standard Numeric       ({len(NUMERIC):2d}): {NUMERIC}")

# Security check: Ensure no columns are lost, unassigned, or duplicated
all_grouped = EXCLUDED + BINARY + NUMERIC_WITH_MISSING + NUMERIC
assert sorted(all_grouped) == sorted(X_fe_preview.columns), "[ERROR] Column grouping mismatch!"

print("\n[OK] All variables explicitly grouped and verified.")

--- 7. VARIABLE ORGANIZATION ---
Excluded               ( 1): ['GameID']
Binary                 ( 1): ['UsesComplexUnits']
Numeric with Missing   ( 4): ['Age', 'log_TotalHours', 'log_HoursPerWeek', 'log_WeeksOfPlay']
Standard Numeric       (18): ['APM', 'AssignToHotkeys', 'UniqueHotkeys', 'NumberOfPACs', 'GapBetweenPACs', 'ActionLatency', 'ActionsInPAC', 'TotalMapExplored', 'WorkersMade', 'UniqueUnitsMade', 'ComplexUnitsMade', 'CognitiveSpeedIndex', 'MinimapUsage', 'ComplexMechanicsRate', 'log_MinimapAttacks', 'log_MinimapRightClicks', 'log_SelectByHotkeys', 'log_ComplexAbilitiesUsed']

[OK] All variables explicitly grouped and verified.


### **8. Pipelines by Variable Type**

In this step, we define the specific sequence of statistical transformations for our numeric variables, routing them based on the groups defined in Step 7 to address the issues identified in the EDA.

*   **For Numeric Variables with Missing Data:** A `SimpleImputer(strategy='median')` is applied first to handle the Missing Not At Random (MNAR) demographic data of League 8 players, intentionally camouflaging them to prevent target leakage. This is followed by a `RobustScaler`.
*   **For Standard Numeric Variables:** Only a `RobustScaler` is applied. We utilize this over `StandardScaler` because our EDA confirmed the presence of valid, extreme high outliers in mechanical telemetry. This scaler uses the median and interquartile range, preventing these true elite outliers from skewing the distribution.

In [6]:
print("--- 8. PIPELINES BY VARIABLE TYPE ---")

# 1. Pipeline for numeric variables WITH missing data (Demographics & Time features)
numeric_missing_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

# 2. Pipeline for standard numeric variables (Telemetry - no missing data)
numeric_pipe = Pipeline(steps=[
    ('scaler', RobustScaler())
])

print("[OK] Sub-pipelines for missing and standard numeric variables defined.")

--- 8. PIPELINES BY VARIABLE TYPE ---
[OK] Sub-pipelines for missing and standard numeric variables defined.


### **9. ColumnTransformer and Full Pipeline Integration**

Here, we integrate all individual components into a single, cohesive architecture.

1. **ColumnTransformer (`preprocessor`)**: Acts as a traffic controller, directing our columns through the specific pipelines defined above:
    *   `NUMERIC_WITH_MISSING`: Routed through median imputation and robust scaling.
    *   `NUMERIC`: Routed through robust scaling only.
    *   `BINARY`: Passed through untransformed (scaling 0s and 1s adds no value).
    *   `EXCLUDED`: Explicitly dropped (e.g., the surrogate key `GameID`).
    *   **Note:** `verbose_feature_names_out=False` is used to preserve the original, clean column names in the final Pandas DataFrame.
2. **Full Pipeline (`prep_pipeline`)**: Chains our custom Feature Engineering function (from Step 6) with the ColumnTransformer. The order is critical: features must be mathematically calculated from raw telemetry *before* imputation and scaling distort their original values.

In [7]:
print("--- 9. FULL PIPELINE INTEGRATION ---")

# 1. Integrate into a ColumnTransformer (The Traffic Controller)
preprocessor = ColumnTransformer(
    transformers=[
        ('num_missing', numeric_missing_pipe, NUMERIC_WITH_MISSING),
        ('num', numeric_pipe, NUMERIC),
        ('bin', 'passthrough', BINARY),
        ('drop_id', 'drop', EXCLUDED)
    ],
    remainder='drop', # Explicitly drops any other undefined columns
    verbose_feature_names_out=False # Prevents adding prefixes to the output column names
)

# 2. Build the Full Pipeline
# Sequence: Engineer features -> Apply routing, imputation & scaling -> Drop unused columns
prep_pipeline = Pipeline(steps=[
    ('feature_engineering', feature_engineer_transformer),
    ('preprocessing', preprocessor)
])

print("[OK] ColumnTransformer configured with specific routing for all feature groups.")
print("[OK] Full pipeline integrated successfully.\n")

# Display the pipeline architecture (renders an HTML diagram in modern Jupyter environments)
prep_pipeline

--- 9. FULL PIPELINE INTEGRATION ---
[OK] ColumnTransformer configured with specific routing for all feature groups.
[OK] Full pipeline integrated successfully.



,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('feature_engineering', ...), ('preprocessing', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function eng...x7ab15a0aef20>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature_names_out` is not None.See ``get_feature_names_out`` for more details... versionadded:: 1.1",None
,"kw_args kw_args: dict, default=

### **10. Pipeline Execution and Structural Validation**

Before exporting our data or moving to the modeling phase, we must validate that our `prep_pipeline` executes flawlessly. 

**Validation Checklist:**
1.  **Data Leakage Prevention:** We apply `.fit_transform()` exclusively to `X_train` to learn the imputation medians and scaling parameters. We then apply `.transform()` to `X_test` using those learned parameters.
2.  **Pandas Output:** Thanks to our earlier Scikit-Learn configuration, the output must remain a Pandas DataFrame with original, clean column names.
3.  **Missing Values Resolution:** All `NaN`s (including the MNAR demographic data and generated infinities) must be fully resolved.

In [8]:
print("--- 10. PIPELINE EXECUTION & VALIDATION ---")

# 1. Fit and transform the training data
X_train_prep = prep_pipeline.fit_transform(X_train)

# 2. Transform the testing data (using parameters learned from X_train)
X_test_prep = prep_pipeline.transform(X_test)

# 3. Structural Verification
print(f"Original X_train features: {X_train.shape[1]}")
print(f"Prepared X_train features: {X_train_prep.shape[1]}")
print(f"Prepared X_test features:  {X_test_prep.shape[1]}")

# 4. Missing Data Verification
missing_train = X_train_prep.isna().sum().sum()
missing_test = X_test_prep.isna().sum().sum()

print(f"\nTotal missing values in prepared X_train: {missing_train}")
print(f"Total missing values in prepared X_test:  {missing_test}")

# Defensive programming assertions
assert missing_train == 0, "[ERROR] Pipeline left missing values in training data!"
assert missing_test == 0, "[ERROR] Pipeline left missing values in testing data!"
assert X_train_prep.shape[1] == X_test_prep.shape[1], "[ERROR] Feature mismatch between train and test sets!"

print("\n[OK] Pipeline validation passed successfully. Data is ready for modeling.")
print("\n--- PREPARED TRAINING DATA PREVIEW ---")
display(X_train_prep.head(3))

--- 10. PIPELINE EXECUTION & VALIDATION ---
Original X_train features: 19
Prepared X_train features: 23
Prepared X_test features:  23

Total missing values in prepared X_train: 0
Total missing values in prepared X_test:  0

[OK] Pipeline validation passed successfully. Data is ready for modeling.

--- PREPARED TRAINING DATA PREVIEW ---


,Age,log_TotalHours,log_HoursPerWeek,log_WeeksOfPlay,APM,AssignToHotkeys,UniqueHotkeys,NumberOfPACs,GapBetweenPACs,ActionLatency,...,UniqueUnitsMade,ComplexUnitsMade,CognitiveSpeedIndex,MinimapUsage,ComplexMechanicsRate,log_MinimapAttacks,log_MinimapRightClicks,log_SelectByHotkeys,log_ComplexAbilitiesUsed,UsesComplexUnits
348,1.4,-0.520557,0.316611,-0.817990,-1.062095,-0.346656,0.000000,-1.677192,2.732269,1.391888,...,-1.000000,0.0,-0.782570,-0.575370,-0.127061,-0.132721,-0.786432,-1.068556,-0.152498,0
278,0.6,0.343194,0.566003,-0.145645,0.639762,-0.035159,0.666667,0.289269,0.087256,-0.478278,...,0.000000,0.0,0.700650,0.060788,-0.127061,1.105699,-0.309451,0.063481,-0.152498,0
2331,1.0,0.386010,1.234476,-0.734547,-0.663364,0.342431,0.000000,-0.303134,-0.211655,0.655620,...,0.333333,0.0,-0.529016,-0.477240,0.335185,-0.112929,-0.584364,-0.455573,0.617547,0


### **11. Final Preparation Summary and Handoff**

Following the project guidelines, we conclude this data preparation phase by summarizing our methodology, limitations, and the exact components that will be transitioned to the modeling phase. Note that no model selection or hyperparameter optimization is performed here.

**1. Summary of Decisions Made:**
*   **Data Splitting:** Applied Stratified Sampling to preserve the extreme class imbalance across the train and test sets.
*   **Imputation:** Used Median Imputation strictly within the pipeline to camouflage the MNAR demographic data of League 8 players, preventing target leakage.
*   **Transformations:** Addressed severe right-skewness using `log1p` combined with domain-specific unit conversions (timestamps to minutes) for in-game action rates.
*   **Scaling:** Selected `RobustScaler` to standardize features while protecting the valid, extreme mechanical outliers that define the professional skill ceiling.

**2. Excluded Variables:**
*   **`GameID`**: Dropped explicitly by the `ColumnTransformer` as it is a surrogate key with no predictive value.
*   **Raw Skewed Variables**: `TotalHours`, `HoursPerWeek`, and raw action rates were dropped dynamically inside the custom feature engineer transformer to avoid redundancy after creating their `log_` transformed counterparts.

**3. Limitations:**
*   The primary limitation is the systematic absence of demographic data (`Age`, `HoursPerWeek`, `TotalHours`) for all League 8 (Professional) players due to anonymization. While median imputation safely prevents target leakage, it forces the future models to rely entirely on in-game mechanical telemetry to identify elite players.

**4. Hypotheses for the Modeling Phase:**
*   We hypothesize that tree-based algorithms will handle the raw, extreme mechanical outliers (`APM`, `ActionLatency`) effectively without strict reliance on our custom scaling.
*   Conversely, we hypothesize that linear models will demonstrate significantly improved accuracy due to our engineered `CognitiveSpeedIndex` (which reduces multicollinearity) and the `log1p` variance stabilization. These approaches will be explicitly compared during modeling.

**5. Objects to be Used in the Modeling Notebook:**
The following objects are now fully configured in memory and will be imported/utilized in the next phase:
*   `X_train`, `X_test`: The stratified feature matrices.
*   `y_train`, `y_test`: The stratified target vectors (`LeagueIndex`).
*   `prep_pipeline`: The fitted master pipeline, ready to transform the data identically and safely for cross-validation and final modeling.
